# Langfuse — Avaliação de Agente LangChain

Notebook para testar e observar o agente localmente com rastreamento completo via **Langfuse**.

**Stack:**
- `create_agent` (LangChain) + `InMemorySaver` (LangGraph)
- `AzureChatOpenAI` como LLM
- `CallbackHandler` do Langfuse para capturar traces end-to-end

## 1. Imports

In [7]:
import os
from pathlib import Path
from typing import Annotated
from langchain_core.messages import AIMessage, AIMessageChunk
from langchain.agents import create_agent
from langchain_core.tools import BaseTool, tool
from langchain_openai import AzureChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langfuse import get_client
from langfuse.langchain import CallbackHandler

from az_aisearch_rag import rag, what_is_langfuse_info

## 2. LLM Factory

Função que cria uma instância do LLM. Cada agente pode receber sua própria instância ou compartilhar uma.

In [8]:
def make_llm() -> AzureChatOpenAI:
    return AzureChatOpenAI(
        azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
        api_version=os.environ["AZURE_OPENAI_API_VERSION"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        stream_usage=True,
    )

## 3. Tools

`@tool` definidos **inline** (necessário para o `CallbackHandler` propagar o tracing).  
A lógica de busca e embedding fica em `tools.py`.

In [9]:
@tool
def rag_tool(
    query: Annotated[
        str, "Pergunta ou termos de busca para consultar a base de conhecimento SSMA."
    ],
) -> str:
    """Busca documentos na base de conhecimento. Use antes de responder ao usuário."""
    return rag(
        index_name="shp-base-conhecimento-ssma-index",
        query=query,
        top_k=5,
        text_field="snippet",
        vector_field="snippet_vector",
        title_field="doc_url",
    )


@tool
def what_is_langfuse() -> str:
    """Retorna um resumo sobre o que é o Langfuse e para que serve."""
    return what_is_langfuse_info()

## 4. Classe Agent

Recebe tudo via argumento — LLM, tools, middleware, system_prompt, checkpointer e name.  
Projetada para instanciação sob demanda em sistemas multi-agent.

In [10]:
def _flush():
    get_client().flush()


class Agent:
    """Agente LangChain configurável via injeção de dependência.

    Parâmetros do construtor (todos via __init__, sem globals):
        system_prompt: Texto do prompt de sistema.
        tools: Lista de BaseTool para o agente.
        llm: Instância de AzureChatOpenAI. Se None, cria via make_llm().
        checkpointer: Checkpointer LangGraph. Se None, usa InMemorySaver().
        middleware: Lista de AgentMiddleware. Se None, sem middleware.
        name: Nome do agente (aparece nos traces).

    Na FastAPI thread_id e user_id são passados por chamada (multi-tenant).
    No notebook são passados no construtor (um Agent por sessão de teste).
    """

    def __init__(
        self,
        system_prompt: str,
        tools: list[BaseTool],
        llm: AzureChatOpenAI | None = None,
        checkpointer=None,
        middleware: list | None = None,
        name: str = "Agent",
    ) -> None:
        self._agent = create_agent(
            model=llm or make_llm(),
            tools=tools,
            system_prompt=system_prompt,
            name=name,
            checkpointer=checkpointer or InMemorySaver(),
            middleware=middleware or [],
        )

    def _build_config(self, thread_id: str, user_id: str) -> dict:
        config: dict = {"configurable": {"thread_id": thread_id, "user_id": user_id}}

        config["callbacks"] = [CallbackHandler()]
        config["metadata"] = {
            "langfuse_session_id": thread_id,
            "langfuse_user_id": user_id,
            "langfuse_tags": ["agent"],
        }
        return config

    def invoke(
        self, user_message: str, *, thread_id: str, user_id: str = "anonymous"
    ) -> str:
        """Retorna a resposta completa do agente (batch)."""
        try:
            result = self._agent.invoke(
                {"messages": [{"role": "user", "content": user_message}]},
                config=self._build_config(thread_id, user_id),
            )
        finally:
            _flush()
        messages = result.get("messages", [])
        last_ai = next(
            (m for m in reversed(messages) if isinstance(m, AIMessage)), None
        )
        return last_ai.content if last_ai else ""

    async def astream(
        self, user_message: str, *, thread_id: str, user_id: str = "anonymous"
    ):
        """Gera chunks de texto da resposta (SSE / streaming)."""
        try:
            async for message, _ in self._agent.astream(
                {"messages": [{"role": "user", "content": user_message}]},
                config=self._build_config(thread_id, user_id),
                stream_mode="messages",
            ):
                if isinstance(message, AIMessageChunk) and message.content:
                    yield message.content
        finally:
            _flush()

## 5. Testes

Cada agente é instanciado explicitamente com seu prompt, tools, middleware e thread_id.  
Cada chamada gera um trace completo: `invoke → agent → tool → LLM`.

In [11]:
_prompts_dir = Path.cwd().parent / "src/agents/prompts"

ssma = Agent(
    system_prompt=(_prompts_dir / "ssma.md").read_text().strip(),
    tools=[rag_tool],
    middleware=[],
    name="SSMA",
)
ssma.invoke("O que é NR-35?", thread_id="ssma-001")

'A **NR-35 – Trabalho em Altura** é a norma regulamentadora que estabelece os requisitos e medidas de proteção para o trabalho executado acima de 2 metros do nível inferior, onde haja risco de queda.  \n\nDe acordo com os **documentos internos da Ultragaz** (como *TB-CO.72.02 – Matriz de Atividades x EPIs* e *PG-CO.81.0004 – Liberação e Acompanhamento de Serviço em Altura*), a aplicação da NR-35 envolve:\n\n### 🔹 Objetivo\nGarantir que toda atividade em altura seja planejada, organizada e executada de forma segura, prevenindo quedas e acidentes.\n\n### 🔹 Pontos principais abordados\n- **Capacitação:** todos os trabalhadores devem ser treinados para execução segura do trabalho em altura.  \n- **Análise de riscos:** antes da execução, deve ser feita uma avaliação conforme as diretrizes da *Análise Preliminar de Risco (PG-CO.72.0007)* ou *Análise Prevencionista da Tarefa (PG-CO.81.0002)*.  \n- **Permissão de trabalho:** a atividade só pode ser iniciada com liberação formal (Permissão de T

In [12]:
ssma_stream = Agent(
    system_prompt=(_prompts_dir / "ssma.md").read_text().strip(),
    tools=[rag_tool],
    name="SSMA",
)
async for chunk in ssma_stream.astream("O que é NR-35?", thread_id="ssma-stream-001"):
    print(chunk, end="", flush=True)

Vou consultar nossa base interna para te trazer uma resposta precisa sobre a **NR-35**, relacionada a trabalho em altura.A **NR-35** (Norma Regulamentadora nº 35) trata das condições e requisitos mínimos para a **realização de trabalhos em altura**, garantindo a **segurança e a saúde dos trabalhadores** envolvidos nessas atividades.

De acordo com os documentos internos da Ultragaz, especialmente o *PG-CO.81.0004 – Liberação e Acompanhamento de Serviço em Altura* e a *Matriz de Atividades x EPIs (TB-CO.72.02)*, a NR-35 se aplica a **toda atividade executada acima de 2 metros** do nível inferior, onde haja risco de queda.

### Pontos principais abordados pela NR-35:
1. **Planejamento e Organização**  
   Toda atividade em altura deve ser previamente analisada por meio da **Análise Preliminar de Risco (APR)** ou **Análise Prevencionista da Tarefa (APT)**, com emissão de **Permissão de Trabalho (PT)** quando aplicável.

2. **Capacitação e Treinamento**  
   O trabalhador só pode executar 